# PlantVillage MSF-Res2Net（Kaggle GPU）

本 Notebook 在 Kaggle 上：

1. **添加数据集**：在 Notebook 右侧 *Add Data* 里加入你的 PlantVillage 数据集（常见为按类别子文件夹存放的 `color` 等）。
2. **添加本仓库**：可用 *File → Add Input* 上传含 `plantvillage_snn` 的仓库压缩包，或使用下方 **git clone**（推荐）。
3. 修改下面 **GitHub 仓库地址** 与 **数据集目录名** 后，从上到下依次运行。

数据加载使用 `data/kaggle_dataloader.py`：在 `/kaggle/input/<你的数据集>/` 下自动查找「类名为子文件夹」的 ImageFolder 根目录（优先名称含 `color`）；若已是该结构也可直接指向该路径。

训练单元中 **ResNet-18** 可通过 `RN18_NEURON` 切换 **MSF / LIF / PLIF**（PLIF 为 SpikingJelly `ParametricLIFNode`，与 LIF 共用 `lif_tau` 作初值）。

In [ ]:
# --- 依赖（Kaggle 已预装 torch/torchvision，补充 SNN 与日志） ---
!pip -q install "spikingjelly>=0.0.0.0.14" tensorboard scikit-learn

In [ ]:
# --- 拉取 GitHub 代码（把 URL 换成你的仓库） ---
import os

os.chdir('/kaggle/working')

GITHUB_URL = "https://github.com/xing11234/plantvillage_snn.git"  # 修改
BRANCH = "master"  # 或 master
CLONE_DIR = "/kaggle/working/plantvillage_snn"


!git clone --depth 1 --branch {BRANCH} {GITHUB_URL} {CLONE_DIR}

os.chdir(CLONE_DIR)


print("Working dir:", os.getcwd())

In [ ]:
# --- 若使用「Upload Code」数据集而非 git：取消注释并指向解压路径 ---
# import os
# os.chdir("/kaggle/input/your-code-dataset/plantvillage_snn")
# print(os.getcwd())

In [ ]:
# --- Kaggle 数据集路径：INPUT 名称与 Data 面板一致 ---
import os

INPUT_NAME = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset"  # 修改：与 /kaggle/input 下文件夹名一致
DATA_ROOT = os.path.join("/kaggle/input", INPUT_NAME)

# 若类文件夹不在 INPUT 根下，保持 True 会自动搜索（如 .../color）
#AUTO_FIND_SUBDIR = True

assert os.path.isdir(DATA_ROOT), f"Missing dataset path: {DATA_ROOT}"
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
# --- 训练（复用 train.py 中的 encode / 单轮训练 / 验证） ---
import os
import sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
from torch import amp as torch_amp
from torch.utils.tensorboard import SummaryWriter

from config import get_config
from data.kaggle_dataloader import get_kaggle_dataloaders
from models.res2net_msf import build_model
from models.spiking_resnet18_backbone import build_spiking_resnet18
from train import encode_batch, evaluate, set_seed, train_one_epoch
from utils.train_utils import SpikeCounter, cosine_lr

# 超参（可按需改）
# 骨干："res2net" = 仓库 MSF-Res2Net（CSA 可关）；"sj_resnet18" = SpikingJelly ResNet-18（轻量，无 CSA）
BACKBONE = "sj_resnet18"  # "res2net" | "sj_resnet18"
# 仅 BACKBONE=="res2net" 时：no_msf / no_attention / ablation_baseline 等（见 config.PRESETS）
PRESET = None
# --- 仅 BACKBONE=="sj_resnet18"：同优化器/数据/编码下切换神经元（消融）---
# "msf" = MSFNode；"lif" = LIFNode；"plif" = ParametricLIFNode（PLIF，可学习膜时间常数，init_tau 用下方 cfg.lif_tau）
RN18_NEURON = "msf"  # "msf" | "lif" | "plif"
EPOCHS = 20  # 全数据正式跑可 20~30；只想验证管线可临时改为 3~5
T_STEPS = 4  # 默认 8 更慢；4 显著减时每 batch 的 SNN 时间展开（与 first_spike 长度一致）
BATCH_SIZE = 64  # MSF 模式通常可 64；LIF（no_msf）在 T4 上会自动降到 32（见下方）。仍 OOM 可手动改 24/16
LR = 0.05  # 峰值略低于默认 0.1，减轻 sj_resnet18 / 中等 batch 下的 val 震荡
# 必须为 False：本模型前向输入是 [T,B,C,H,W]，dim0 是时间步 T。nn.DataParallel 会在 dim0 上切「batch」，
# 实际会把 T 拆到多卡，前向语义错误，验证集准确率会接近随机（~1/num_classes），训练集仍可能虚高。
USE_DATA_PARALLEL = False
if torch.cuda.device_count() > 1:
    print(
        "NOTE: 未启用 DataParallel（与 [T,B,...] 布局不兼容）。Kaggle 多 GPU 时仅用 cuda:0；"
        "要利用多卡需 DDP 或把张量改成以 B 为第 0 维并改模型。",
        flush=True,
    )

_cfg_kw = dict(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, T=T_STEPS)
if BACKBONE == "sj_resnet18":
    _rn = RN18_NEURON.strip().lower()
    if _rn == "msf":
        cfg = get_config(PRESET, **_cfg_kw, use_msf=True, lif_variant="lif")
    elif _rn == "lif":
        cfg = get_config(PRESET, **_cfg_kw, use_msf=False, lif_variant="lif")
    elif _rn == "plif":
        cfg = get_config(PRESET, **_cfg_kw, use_msf=False, lif_variant="plif")
    else:
        raise ValueError("RN18_NEURON 必须是 msf / lif / plif")
else:
    cfg = get_config(PRESET, **_cfg_kw)
# 与 LIF 公平对比：PLIF 的 init_tau 默认等于 TrainConfig.lif_tau（可在下面改 cfg.lif_tau）
# 缓解过拟合、平滑决策边界（0 关闭；与 ANN 对比时可两边同设为 0.05–0.1）
cfg.label_smoothing = 0.05
if BACKBONE == "sj_resnet18":
    cfg.save_name = f"best_rn18_{RN18_NEURON.strip().lower()}_T{cfg.T}.pt"

# --- MSF 代理梯度：换 arctan / sigmoid 时常与 α、学习率联合调 ---
# cfg.msf_surrogate = "arctan"   # 或 "sigmoid"
# cfg.msf_surrogate_alpha = 3.0  # 光滑 surrogate 的宽度；arctan 可试 2~5；sigmoid 可试 0.5~2（过大窗窄易弱梯度）
# α 越大通常越「尖」；过小则梯度太平。rect 对 α 几乎不敏感（仍参与 rect 窗半宽）。
# 光滑 surrogate 的平均反向幅度常低于 rect：若欠拟合可略增 cfg.lr（如 0.08~0.1）或略增 α，建议与 rect 分开报告超参。
# 默认 warmup_epochs=5：若总 epoch 很少，会整段都在「升温」、几乎没有 cosine 尾部
cfg.warmup_epochs = min(cfg.warmup_epochs, max(1, cfg.epochs // 2))
# LIF / PLIF 模式（use_msf=False）多步前向峰值显存常高于 MSF；T4 15GB + batch=64 易 CUDA OOM
if not cfg.use_msf and cfg.batch_size > 32:
    print(
        "NOTE: use_msf=False (LIF/PLIF) — lowering batch_size",
        cfg.batch_size,
        "-> 32 to reduce OOM on ~15GB GPUs.",
        flush=True,
    )
    cfg.batch_size = 32
# --- 缩短墙钟（可选，折中精度/日志；在 get_kaggle_dataloaders 之前改 image_size）---
# cfg.spike_counter_enabled = False  # 关脉冲统计，省一点每 step 开销
# cfg.image_size = 192  # 比 224 快一截，精度通常略降；正式报告可改回 224
cfg.num_workers = 2
cfg.batch_log_interval = 50  # 每 N 个 batch 打印一次；改为 0 可关闭
cfg.checkpoint_dir = "/kaggle/working/checkpoints"
cfg.log_dir = "/kaggle/working/runs"
cfg.viz_dir = "/kaggle/working/viz"
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print(
    f"Time steps T={cfg.T} (each batch: first_spike_coding -> [T,B,3,H,W], backbone runs T steps; "
    f"larger T is slower). warmup_epochs={cfg.warmup_epochs}, cosine to epoch {cfg.epochs - 1}.",
    flush=True,
)

# 若未先运行「数据集路径」单元，与默认行为一致（在 INPUT 下自动查找 ImageFolder 根）
if "AUTO_FIND_SUBDIR" not in globals():
    AUTO_FIND_SUBDIR = True

train_loader, val_loader, num_classes = get_kaggle_dataloaders(
    DATA_ROOT,
    image_size=cfg.image_size,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=0.2,
    seed=cfg.seed,
    auto_find_subdir=AUTO_FIND_SUBDIR,
)
cfg.num_classes = num_classes
print("num_classes:", num_classes, "train batches:", len(train_loader), "val batches:", len(val_loader))

if BACKBONE == "sj_resnet18":
    model = build_spiking_resnet18(cfg).to(device)
    _lab = (
        "MSF"
        if cfg.use_msf
        else ("PLIF (ParametricLIF)" if getattr(cfg, "lif_variant", "lif") == "plif" else "LIF")
    )
    print("Backbone: SpikingJelly ResNet-18 +", _lab, flush=True)
else:
    model = build_model(cfg).to(device)
    print("Backbone: MSF-Res2Net", flush=True)
if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print(f"nn.DataParallel enabled on {torch.cuda.device_count()} GPUs", flush=True)

def _unwrap(m):
    return m.module if isinstance(m, torch.nn.DataParallel) else m

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=cfg.lr,
    momentum=cfg.momentum,
    weight_decay=cfg.weight_decay,
    nesterov=True,
)
scaler = torch_amp.GradScaler(enabled=cfg.amp and device.type == "cuda")
writer = SummaryWriter(log_dir=os.path.join(cfg.log_dir, "kaggle"))

spike_counter = (
    SpikeCounter(_unwrap(model), enabled=cfg.spike_counter_enabled)
    if cfg.spike_counter_enabled
    else None
)

print(
    f"Starting training: {cfg.epochs} epochs, {len(train_loader)} train batches / epoch, "
    f"{len(val_loader)} val batches",
    flush=True,
)
best_acc = 0.0
for epoch in range(cfg.epochs):
    lr_now = cosine_lr(epoch, cfg.lr, cfg.warmup_epochs, cfg.epochs, cfg.min_lr)
    for pg in optimizer.param_groups:
        pg["lr"] = lr_now

    print(f"\n--- Epoch {epoch + 1}/{cfg.epochs}  lr={lr_now:.5f} ---", flush=True)
    train_loss, train_acc, train_spike = train_one_epoch(
        model, train_loader, optimizer, scaler, device, cfg, spike_counter
    )
    print("  validating...", flush=True)
    val_acc, val_spike = evaluate(model, val_loader, cfg.T, device, cfg.amp, spike_counter=spike_counter)

    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("acc/train", train_acc, epoch)
    writer.add_scalar("acc/val", val_acc, epoch)
    if spike_counter is not None:
        writer.add_scalar("spike/train", train_spike, epoch)
        writer.add_scalar("spike/val", val_spike, epoch)

    print(
        f"Epoch {epoch+1}/{cfg.epochs} lr={lr_now:.5f} loss={train_loss:.4f} "
        f"train_acc={train_acc*100:.2f}% val_acc={val_acc*100:.2f}%"
        + (f" spikes={train_spike:.4f}/{val_spike:.4f}" if spike_counter else "")
    )

    if val_acc > best_acc:
        best_acc = val_acc
        ckpt_path = os.path.join(cfg.checkpoint_dir, cfg.save_name)
        torch.save(
            {
                "model": _unwrap(model).state_dict(),
                "cfg": cfg.to_dict(),
                "epoch": epoch,
                "val_acc": val_acc,
            },
            ckpt_path,
        )
        print("  saved", ckpt_path)

writer.close()
if spike_counter is not None:
    spike_counter.remove()

print(f"Done. Best val acc = {best_acc*100:.2f}%")

## 可选：下载权重与 TensorBoard

- 权重：`/kaggle/working/checkpoints/best_msf_res2net.pt`，在 Kaggle 界面 *Output* 打包下载。
- 日志：`/kaggle/working/runs`，可下载后在本地运行 `tensorboard --logdir runs`。